# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 1.38e-13 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.49e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405053e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 5.34058e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797543969221541 - 0.09754507709792246*y - 0.4175950563713448*x + 2.1931518901396174*y^2 - 0.1269606244678339*x*y - 0.8025867547845221*x^2)^2 + (0.9043573827480671 - 0.6455003192597278*y + 0.2742159536999832*x - 0.10770174782358159*y^2 - 1.2293138068800717*x*y - 0.9300160939847*x^2)^2 + (-0.5739168240154698 - 1.38535050221548*y - 0.6219655378569912*x - 0.2040924100740057*y^2 - 0.15182215880489783*x*y + 0.4443832632766482*x^2)^2 + (0.5703021360913615 - 0.5184093413621602*y + 0.3436728787624473*x + 0.2555950605470665*y^2 + 0.7618441294650047*x*y - 0.02090567376761348*x^2)^2 + (0.041400837217767876 + 0.04911040272637946*y - 0.49990658891857337*x - 0.24792145939081137*y^2 + 0.2972445689917418*x*y - 0.5054184235227418*x^2)^2 + (-0.2524531788141328 - 0.06336647012459033*y + 0.253942572224051*x - 0.10018744153048117*y^2 + 0.059601795985895156*x*y - 0.19381241255811912*x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Computing a Gröbner basis before passing such a domain to SumOfSquares
can take a non-negligible amount of time for large systems of equalities.
So this is not done by default in SumOfSquares but we can request it
explicitly with `SemialgebraicSets.compute_gröbner_basis!`.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  189.305 μs (7906 allocations: 296.12 KiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  369.892 μs (11632 allocations: 420.13 KiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
To reduce the polynomial modulo the full ideal, we compute the Gröbner
basis before passing the domain to the SOS constraint.

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
SemialgebraicSets.compute_gröbner_basis!(S.I)
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 2.23e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 2.78e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  6.8808202e-01 
Iter:  2 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  4.8373997e-01 
Iter:  3 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  2.9627513e-01 
Iter:  4 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  5.2454909e-02 
Iter:  5 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  5.5435833e-03 
Iter:  6 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  7.7648302e-04 
Iter:  7 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  1.0529797e-04 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 1.36089e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(8.226181798637134e-18*y + 3.2323778234884603e-47*x - 1.54136861601437*y^2 + 4.190896004295869*x*y - 2.850760091768449*x^2)^2 + (-9.483969103911849e-18*y - 2.963664775822692e-47*x + 2.914444822095325*y^2 - 1.298256775779121*x*y - 3.484366483425882*x^2)^2 + (-1.7667618375872994e-16*y + 2.2979258587880002e-15*x + 0.13504353593511692*y^2 + 0.10092370670562986*x*y + 0.07535144458687125*x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

3.569168504906172482804275858096587352263621794871794871794871794871844474404006e-9y + 4.621392226165601012046233965800358698918269230769230769230769230769209510969725e-9x + 3.137208964449669513882894275951162931241953556178502394209295783499461386499596e-32y² - 8.119775425823320796209614848882681263704781172905354295746975090636287843142034e-31xy + 5.280463252486568246166715403610653061019745573940825712682903626870516098569162e-30x² + 6.661338147750939242541790008544921875e-16xy² + 4.563942860592344256739087030045597617732721805114404212133649707539007067680359e-16x²y + 1.208460299350323251788862622701204740084134615384615384615384615384604755484856e-8y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

3.569173947845718850238391986260047325721153846153846153846153846153851468411411e-9y + 4.621382661167235010697578008358295147235576923076923076923076923076925734205689e-9x + 9.999999999999983665613357240731229325092499493621289730072021484375e-9y² + 5.600143501331754543683925959192909924541807682548193576191626214017455669958467e-47xy + 9.999999999999983665613357240731229325092499493621289730072021484375e-9x² + 3.390715222890391953456232526597319639925440237782083841011626645922660827636719e-29x²y + 1.208460313014606631790789274068979116586538461538461538461538461538459412635436e-8y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*